In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.model_selection import train_test_split
import xgboost as xgb
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.impute import SimpleImputer

In [18]:
df = pd.read_csv("../../PreviousDataset.csv")

In [19]:
numeric_cols = ['SSTA', 'ClimSST', 'Depth', 'Longitude_Degrees',
                'Latitude_Degrees', 'Average_Bleaching', 'Windspeed',
                'TSA', 'TSA_DHW', 'SSTA_DHW', 'SSTA_Minimum']
for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df['Year'] = df['Date'].dt.year

# Filter
df = df[df['SSTA'] > 0]
df = df[df['Year'] >= 2003]

In [20]:
df = df.sort_values(['Latitude_Degrees', 'Longitude_Degrees', 'Date'])
df['SSTA_Rolling_Mean'] = df.groupby(['Latitude_Degrees', 'Longitude_Degrees'])['SSTA'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
df['TSA_Rolling_Mean'] = df.groupby(['Latitude_Degrees', 'Longitude_Degrees'])['TSA'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

drop_cols = ['City_Town', 'City_Town_2', 'City_Town_3', 'Ocean', 'Realm', 'Ecoregion', 'ID', 'Country_Name', 'State_Island_Province', 'Date2', 'SSTA_Mean']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

# Month encoding
df['Month'] = df['Date'].dt.month
df['Month_sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_cos'] = np.cos(2 * np.pi * df['Month'] / 12)
df = df.drop(columns=['Month', 'Date', 'Year'])

# Train

In [21]:
X = df.drop("Average_Bleaching", axis=1)
Y = df["Average_Bleaching"]

In [22]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [23]:
print(X_train_raw.isna().any().any())
print(X_test_raw.isna().any().any())

True
True


In [24]:
# Impute
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train_raw)
X_test_imp = imputer.transform(X_test_raw)


In [25]:
scaler = StandardScaler()
X_train_ss = scaler.fit_transform(X_train_imp)
X_test_ss = scaler.transform(X_test_imp)

In [26]:
pca = PCA(n_components=0.95) # Increased variance capture
X_train_pca = pca.fit_transform(X_train_ss)
X_test_pca = pca.transform(X_test_ss)

## OpTuna

In [28]:
import optuna
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.model_selection import cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

In [29]:
def rf_objective(trial):

    model = RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 300),
        max_depth=trial.suggest_int("max_depth", 5, 40),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 5),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        n_jobs=-1,
        random_state=42
    )

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        # ('selector', SelectKBest(score_func=f_regression,
        #                          k=trial.suggest_int("k_features", 5, X.shape[1]))),
        ('pca', PCA(
            n_components= 0.95 #trial.suggest_float("pca_variance", 0.90, 0.99)
        )),
        ('model', model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train_raw,
        y_train,
        cv=4,
        scoring='r2',
        n_jobs=-1
    )

    return scores.mean()

rf_study = optuna.create_study(direction="maximize")

rf_study.optimize(rf_objective, n_trials=50)

print("========== RF BEST ==========")
print(rf_study.best_params)
print(f"Best R²: {rf_study.best_value:.4f}")

[I 2026-05-04 17:04:35,457] A new study created in memory with name: no-name-8d4c49b5-dbb6-4d14-9b13-e001c531a1e1
[I 2026-05-04 17:04:37,714] Trial 0 finished with value: 0.2145077602905933 and parameters: {'n_estimators': 217, 'max_depth': 18, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.2145077602905933.
[I 2026-05-04 17:04:39,152] Trial 1 finished with value: 0.17592149266083498 and parameters: {'n_estimators': 148, 'max_depth': 7, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.2145077602905933.
[I 2026-05-04 17:04:40,557] Trial 2 finished with value: 0.2075331593403271 and parameters: {'n_estimators': 81, 'max_depth': 10, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 0 with value: 0.2145077602905933.
[I 2026-05-04 17:04:41,183] Trial 3 finished with value: 0.24467745767565593 and parameters: {'n_estimators': 85, 'max_depth': 21, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 3 with value:

========== RF BEST ==========
{'n_estimators': 300, 'max_depth': 36, 'min_samples_leaf': 1, 'max_features': 'log2'}
Best R²: 0.2586


In [30]:
def xgb_objective(trial):

    model = xgb.XGBRegressor(
        n_estimators=trial.suggest_int("n_estimators", 50, 400),
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.1),
        max_depth=trial.suggest_int("max_depth", 3, 15),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        n_jobs=-1,
        random_state=42
    )

    pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        # ('selector', SelectKBest(
        #     score_func=f_regression,
        #     k=trial.suggest_int("k_features", 5, X.shape[1])
        # )),
        ('pca', PCA(
            n_components= 0.95 #trial.suggest_float("pca_variance", 0.90, 0.99)
        )),
        ('model', model)
    ])

    scores = cross_val_score(
        pipeline,
        X_train_raw,
        y_train,
        cv=4,
        scoring='r2',
        n_jobs=-1
    )

    return scores.mean()

xgb_study = optuna.create_study(direction="maximize")

xgb_study.optimize(xgb_objective, n_trials=50)

print("========== XGB BEST ==========")
print(xgb_study.best_params)
print(f"Best R²: {xgb_study.best_value:.4f}")

[I 2026-05-04 17:05:45,890] A new study created in memory with name: no-name-56d790f9-0fda-43db-9952-66492cd64afb
[I 2026-05-04 17:05:46,802] Trial 0 finished with value: 0.23248774216482476 and parameters: {'n_estimators': 306, 'learning_rate': 0.08117128035830765, 'max_depth': 5, 'subsample': 0.986138597702884, 'colsample_bytree': 0.8014183766863512}. Best is trial 0 with value: 0.23248774216482476.
[I 2026-05-04 17:05:50,821] Trial 1 finished with value: 0.2460849276378998 and parameters: {'n_estimators': 182, 'learning_rate': 0.010619129662926757, 'max_depth': 13, 'subsample': 0.8458164060735132, 'colsample_bytree': 0.6445608217114575}. Best is trial 1 with value: 0.2460849276378998.
[I 2026-05-04 17:05:51,870] Trial 2 finished with value: 0.22137716529549734 and parameters: {'n_estimators': 55, 'learning_rate': 0.05821995037504924, 'max_depth': 9, 'subsample': 0.9507882736159236, 'colsample_bytree': 0.9202868066512353}. Best is trial 1 with value: 0.2460849276378998.
[I 2026-05-04

========== XGB BEST ==========
{'n_estimators': 314, 'learning_rate': 0.03891061952824178, 'max_depth': 11, 'subsample': 0.5000520316952043, 'colsample_bytree': 0.7715187143689911}
Best R²: 0.2782


## Start!

In [31]:
xgb_model = xgb.XGBRegressor(
    n_estimators=314, 
    learning_rate=0.039, 
    max_depth=11, 
    subsample=0.5, 
    colsample_bytree=0.77, 
    n_jobs=-1,
    random_state=42
)

rf_model = RandomForestRegressor(
    n_estimators=300, 
    max_depth=36, 
    min_samples_leaf=1,
    max_features='log2',
    n_jobs=-1, 
    random_state=42
)
# {'n_estimators': 300, 'max_depth': 36, 'min_samples_leaf': 1, 'max_features': 'log2'}

In [32]:
voting = VotingRegressor([('xgb', xgb_model), ('rf', rf_model)])
voting.fit(X_train_pca, y_train)

# Evaluation
preds = voting.predict(X_test_pca)
r2 = r2_score(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"\n--- Leakage-Free Results ---")
print(f"R-squared: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")


--- Leakage-Free Results ---
R-squared: 0.4209
RMSE: 7.3713


In [33]:
from sklearn.model_selection import KFold, cross_val_score
import numpy as np

# %%
# K-Fold Validation 

n_splits = 4
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

r2_scores   = []
rmse_scores = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    X_train_raw, X_test_raw = X.iloc[train_idx], X.iloc[test_idx]
    y_train,     y_test     = Y.iloc[train_idx], Y.iloc[test_idx]

    # Impute
    imputer     = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train_raw)
    X_test_imp  = imputer.transform(X_test_raw)

    # Scale
    scaler      = StandardScaler()
    X_train_ss  = scaler.fit_transform(X_train_imp)
    X_test_ss   = scaler.transform(X_test_imp)

    # PCA
    pca         = PCA(n_components=0.95)
    X_train_pca = pca.fit_transform(X_train_ss)
    X_test_pca  = pca.transform(X_test_ss)

    # Train
    voting      = VotingRegressor([('xgb', xgb_model), ('rf', rf_model)])
    voting.fit(X_train_pca, y_train)

    # Evaluate
    preds = voting.predict(X_test_pca)
    r2_scores.append(r2_score(y_test, preds))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, preds)))

    print(f"Fold {fold}  →  R²: {r2_scores[-1]:.4f}  |  RMSE: {rmse_scores[-1]:.4f}")

print(f"\n--- {n_splits}-Fold Summary ---")
print(f"R²   mean: {np.mean(r2_scores):.4f}  ±  {np.std(r2_scores):.4f}")
print(f"RMSE mean: {np.mean(rmse_scores):.4f}  ±  {np.std(rmse_scores):.4f}")

Fold 1  →  R²: 0.3753  |  RMSE: 7.4076
Fold 2  →  R²: 0.3393  |  RMSE: 8.8395
Fold 3  →  R²: 0.2061  |  RMSE: 8.6765
Fold 4  →  R²: 0.3354  |  RMSE: 6.8119

--- 4-Fold Summary ---
R²   mean: 0.3141  ±  0.0642
RMSE mean: 7.9339  ±  0.8526
